# Typed Wire Layer

> Typed data transfer at the worker boundary — the zero-copy `FileBackedDTO`
> protocol and (stage 2 of the post-pass-2 execution sequence) the typed
> result envelope that carries task results across the worker HTTP/JSON
> boundary, retiring the per-consumer `field_of` dict-or-object tolerance
> helpers (pass-2 Thread 3; evidence E5/D10/C7).

In [ ]:
#| default_exp core.wire

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import dataclasses
import logging
from typing import Any, Callable, Dict, Protocol, runtime_checkable

_logger = logging.getLogger(__name__)

## FileBackedDTO Protocol

The `FileBackedDTO` protocol defines objects that can serialize themselves to disk for zero-copy transfer between Host and Worker processes. When the Proxy detects an argument implementing this protocol, it calls `to_temp_file()` and sends the file path instead of the data.

In [ ]:
#| export
@runtime_checkable
class FileBackedDTO(Protocol):
    """Protocol for Data Transfer Objects that serialize to disk for zero-copy transfer."""
    
    def to_temp_file(self) -> str: # Absolute path to the temporary file
        """Save the data to a temporary file and return the absolute path."""
        ...

In [ ]:
# Test FileBackedDTO protocol detection
import tempfile

class MockAudioData:
    """Example class implementing FileBackedDTO."""
    
    def __init__(self, data: bytes):
        self._data = data
    
    def to_temp_file(self) -> str:
        """Save to temp file and return path."""
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
            f.write(self._data)
            return f.name

# Check if it implements the protocol
audio = MockAudioData(b"fake audio data")
print(f"MockAudioData implements FileBackedDTO: {isinstance(audio, FileBackedDTO)}")
print(f"Temp file path: {audio.to_temp_file()}")

# A regular string does not implement the protocol
print(f"str implements FileBackedDTO: {isinstance('hello', FileBackedDTO)}")

## Typed result envelope

The typed result envelope is how task results keep their type across the
worker HTTP/JSON boundary. Today a worker serializes `execute()` results via
`EnhancedJSONEncoder` (`dataclasses.asdict`) — the type information is
destroyed at the boundary, every consumer re-extracts fields from anonymous
dicts, and each workflow core re-pays the dict-or-object tolerance tax
(`field_of`; evidence E5/D10/C7 + 10 inline e2e-script copies).

Design (serialization precedent: the `cjm-context-graph-primitives`
tagged-dict law, stage-1 ledger P5/P6):

- **Dict-primary, self-describing**: `{"__wire__": "<kind>", "data": {...}}`.
- **Encode is exact-type**: only DTO classes registered via `@wire_type`
  get the envelope; everything else (plain dicts from dispatcher plugins,
  primitives, unregistered dataclasses) passes through to the
  `EnhancedJSONEncoder` path exactly as before — adoption is per-DTO,
  not flag-day.
- **Decode is strict on known kinds, tolerant on unknown kinds**: a
  registered kind reconstructs through the DTO's `from_dict` (a missing
  required field raises loudly); an unknown kind passes through unchanged
  with the envelope intact, so a host without the result's interface
  library degrades to today's dict behavior instead of failing.
- **One deliberate divergence from the P6 law**: results are a transport
  TERMINUS (consumed, not re-persisted), so extra keys on a known kind are
  dropped with a debug log rather than raising — version skew between a
  worker env (snapshot installs) and a host env must not hard-fail a run.
  Surface-drift visibility is the manifest live-companion's job, not the
  result path's.

Direction note: this is the RESULT direction (worker → host). Typed task
ARGUMENTS (host → worker) ride the CR-17 adapter routing work (stage 4);
until then `_prepare_payload` / `FileBackedDTO` cover the input direction.

In [ ]:
#| export
# Wire format constants + registries. kind -> class for decode;
# class -> kind (exact type) for encode.
WIRE_KIND_KEY = "__wire__"
WIRE_DATA_KEY = "data"

_WIRE_TYPES: Dict[str, type] = {}
_WIRE_KINDS: Dict[type, str] = {}

In [ ]:
#| export
def flat_from_dict(
    cls,      # The wire DTO dataclass being reconstructed
    d: dict,  # The envelope's "data" payload
):            # An instance of `cls`
    """Default reconstruction for FLAT wire DTOs (no nested-DTO fields).

    Filters the payload to the dataclass's declared fields (unknown extras
    are dropped with a debug log — transport-terminus tolerance, see the
    envelope design note) and lets the constructor enforce required fields
    (a missing required field raises TypeError loudly). DTOs with nested
    DTO fields (e.g. a result carrying a list of typed items) must define
    their own `from_dict` classmethod; `@wire_type` only attaches this
    default when the class has none.
    """
    names = {f.name for f in dataclasses.fields(cls)}
    extras = set(d) - names
    if extras:
        _logger.debug("wire: dropping unknown keys %s for %s",
                      sorted(extras), cls.__name__)
    return cls(**{k: v for k, v in d.items() if k in names})

In [ ]:
#| export
def wire_type(
    kind: str  # Stable wire discriminator, e.g. "transcription.result"
) -> Callable[[type], type]:  # Class decorator
    """Register a dataclass as a typed wire DTO under `kind`.

    - The class must be a dataclass (encode falls back to
      `dataclasses.asdict` when it defines no `to_dict`).
    - If the class defines no `from_dict`, the flat default
      (`flat_from_dict`) is attached; nested DTOs define their own.
    - Re-registering the same LOGICAL class (qualname match; the module is
      ignored because nbdev's literate workflow defines each class twice —
      in-notebook `__main__` + the exported module) replaces the decode
      entry; a DIFFERENT class claiming an already-registered kind raises
      ValueError.
    """
    def decorator(cls: type) -> type:
        if not dataclasses.is_dataclass(cls):
            raise TypeError(f"@wire_type({kind!r}) requires a dataclass, got {cls!r}")
        existing = _WIRE_TYPES.get(kind)
        if existing is not None and existing.__qualname__ != cls.__qualname__:
            raise ValueError(
                f"wire kind {kind!r} already registered to "
                f"{existing.__module__}.{existing.__qualname__}")
        # The MODULE is deliberately NOT compared: nbdev's literate workflow
        # defines the same logical class twice (in-notebook `__main__` + the
        # exported module import), and both can coexist in one process (e.g.
        # nbdev-test --n_workers 1 runs sibling notebooks in one worker). Both
        # registrations stay encodable (the prior class keeps its _WIRE_KINDS
        # entry); decode resolves to the most recently registered definition.
        if getattr(cls, "from_dict", None) is None:
            cls.from_dict = classmethod(flat_from_dict)
        _WIRE_TYPES[kind] = cls
        _WIRE_KINDS[cls] = kind
        return cls
    return decorator

In [ ]:
#| export
def wire_encode(
    obj: Any  # A task result (any shape)
) -> Any:     # Tagged envelope dict for registered DTOs; `obj` unchanged otherwise
    """Wrap a registered wire DTO in its tagged envelope (worker side).

    Exact-type lookup: subclasses are NOT encoded under the parent's kind
    (they pass through unregistered, preserving today's behavior).
    Payload preference: the DTO's own `to_dict()` when defined, else
    `dataclasses.asdict` (recursive — nested dataclasses flatten).
    """
    kind = _WIRE_KINDS.get(type(obj))
    if kind is None:
        return obj
    to_dict = getattr(obj, "to_dict", None)
    data = to_dict() if callable(to_dict) else dataclasses.asdict(obj)
    return {WIRE_KIND_KEY: kind, WIRE_DATA_KEY: data}


In [ ]:
#| export
def wire_decode(
    obj: Any  # A JSON-decoded response body (any shape)
) -> Any:     # The typed DTO for known kinds; `obj` unchanged otherwise
    """Reconstruct a typed result from its tagged envelope (host side).

    Known kind -> the registered class's `from_dict` (strict: a missing
    required field raises). Unknown kind -> the dict passes through
    UNCHANGED with the envelope intact (tolerant degradation for hosts
    without the result's interface library). Untagged values pass through.
    """
    if not (isinstance(obj, dict) and WIRE_KIND_KEY in obj):
        return obj
    cls = _WIRE_TYPES.get(obj.get(WIRE_KIND_KEY))
    if cls is None:
        return obj
    return cls.from_dict(obj.get(WIRE_DATA_KEY) or {})

In [ ]:
# Wire-format executable tests (the cross-boundary payload discipline):
# round-trips simulate the real boundary — encode -> json.dumps -> json.loads
# -> decode — never in-memory shortcuts.
import json as _json
from dataclasses import dataclass, field
from typing import List, Optional


@wire_type("test.flat")
@dataclass
class _FlatResult:
    text: str
    confidence: Optional[float] = None
    metadata: dict = field(default_factory=dict)


@dataclass
class _Item:
    text: str
    start_time: float
    end_time: float


@wire_type("test.nested")
@dataclass
class _NestedResult:
    items: List[_Item]
    metadata: dict = field(default_factory=dict)

    @classmethod
    def from_dict(cls, d: dict) -> "_NestedResult":
        return cls(items=[_Item(**i) for i in d.get("items", [])],
                   metadata=d.get("metadata", {}) or {})


def _roundtrip(obj):
    return wire_decode(_json.loads(_json.dumps(wire_encode(obj))))


# 1. Flat DTO round-trips typed through the simulated boundary.
flat = _FlatResult(text="hello", confidence=0.9, metadata={"lang": "en"})
back = _roundtrip(flat)
assert isinstance(back, _FlatResult) and back == flat, back

# 2. Nested DTO round-trips through its custom from_dict (items re-typed).
nested = _NestedResult(items=[_Item("a", 0.0, 1.0), _Item("b", 1.0, 2.0)])
nback = _roundtrip(nested)
assert isinstance(nback, _NestedResult) and isinstance(nback.items[0], _Item)
assert nback == nested

# 3. Unregistered objects pass through encode unchanged (plain dicts /
#    untyped plugin results keep today's behavior).
plain = {"rows": [[1, 2]], "count": 2}
assert wire_encode(plain) is plain
assert wire_decode(plain) is plain

# 4. Unknown kind passes through decode UNCHANGED, envelope intact
#    (tolerant degradation; lossless if ever re-serialized).
foreign = {WIRE_KIND_KEY: "some.future/kind", WIRE_DATA_KEY: {"x": 1}}
assert wire_decode(foreign) is foreign

# 5. Exact-type encode: a subclass is NOT encoded under the parent's kind.
@dataclass
class _FlatSubclass(_FlatResult):
    pass

sub = _FlatSubclass(text="sub")
assert wire_encode(sub) is sub

# 6. Duplicate-kind guard: a DIFFERENT class claiming a taken kind raises;
#    re-registering the same logical class (same qualname; module ignored — nbdev defines classes twice) replaces quietly.
try:
    @wire_type("test.flat")
    @dataclass
    class _Imposter:
        y: int = 0
    raise AssertionError("duplicate kind must raise")
except ValueError:
    pass

# 7. Transport-terminus tolerance: extras dropped (debug-logged), missing
#    required raises loudly.
tolerant = wire_decode({WIRE_KIND_KEY: "test.flat",
                        WIRE_DATA_KEY: {"text": "t", "new_field_from_future": 1}})
assert tolerant == _FlatResult(text="t")
try:
    wire_decode({WIRE_KIND_KEY: "test.flat", WIRE_DATA_KEY: {"confidence": 0.5}})
    raise AssertionError("missing required field must raise")
except TypeError:
    pass

print("typed wire envelope tests OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()